# THEOS-2 Road Surface, Centerline และ Boundary — Full-scene Prediction

โน้ตบุ๊กนี้ใช้ checkpoint จากไฟล์ Fine-tuning เพื่อประมวลผลภาพ THEOS-2 ขนาดใหญ่แบบ sliding window และผลิตผลลัพธ์ GIS

## Outputs ต่อภาพ

```text
geotiff/
├── road_surface_probability.tif
├── road_centerline_probability.tif
├── road_boundary_probability.tif
├── road_surface_mask.tif
├── road_centerline_mask.tif
└── road_boundary_mask.tif

vector/
├── road_surface.gpkg
├── road_centerline.gpkg
└── road_boundary.gpkg
```

`road_centerline.gpkg` จะมีค่า `width_med_m` และ `width_mean_m` แบบประมาณจาก distance transform เป็น attribute เสริม โดยไม่มี road graph, node หรือ edge output


## Session 1 — ติดตั้ง Library

ไม่ติดตั้ง PyTorch ใหม่ และติดตั้งเฉพาะ package ที่ยังไม่มี เพื่อรักษา CUDA environment ของ Colab


In [ ]:
# ============================================================
# 1 — INSTALL ONLY MISSING PACKAGES
# ============================================================

import sys
import subprocess
import importlib.util

PACKAGE_MAP = {
    "rasterio": "rasterio",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "pyogrio": "pyogrio",
    "skimage": "scikit-image",
    "scipy": "scipy",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "fiona": "fiona",
}
missing = [pkg for module, pkg in PACKAGE_MAP.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *missing])

import torch
import torchvision
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA ready:", torch.cuda.is_available())


## Session 2 — Mount Drive และกำหนด Paths

โดยปกติแก้เฉพาะ cell นี้:

- `IMAGE_SOURCES`
- `CHECKPOINT_SOURCES`
- `OUTPUT_ROOT`

ระบบจะค้นหา checkpoint ที่ชื่อมีคำว่า `best` ก่อน แล้วเลือก epoch/score สูงสุดจาก metadata


In [ ]:
# ============================================================
# 2 — DRIVE, INPUTS AND OUTPUTS
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = (
    DRIVE_ROOT
    / "GeoFM_Projects"
    / "Road_Extraction_THEOS2"
)

IMAGE_SOURCES = [
    DRIVE_ROOT / "THEOS2_images",
]

CHECKPOINT_SOURCES = [
    PROJECT_ROOT / "checkpoints",
]

OUTPUT_ROOT = PROJECT_ROOT / "predictions"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT :", OUTPUT_ROOT)


## Session 3 — Prediction และ Post-processing Configuration

ค่า threshold สามารถปรับหลังดูผลตัวอย่างได้โดยไม่ต้องฝึกใหม่

- `SURFACE_THRESHOLD` มีผลต่อความหนา/ความครบถ้วนของถนน
- Centerline สุดท้ายใช้ hybrid ระหว่าง centerline head และ skeleton ของ surface
- Boundary สุดท้ายใช้ geometric boundary ของ surface ร่วมกับ boundary head


In [ ]:

# ============================================================
# 3 — PREDICTION CONFIGURATION (CENTERLINE V2)
# ============================================================

PATCH_SIZE_OVERRIDE = None
STRIDE = 384
BATCH_SIZE = 6
USE_AMP = True

SURFACE_THRESHOLD = 0.50
# This threshold is for diagnostic preview/confidence only.  It never gates the surface skeleton.
CENTERLINE_PROBABILITY_THRESHOLD = 0.35
BOUNDARY_THRESHOLD = 0.42

VALID_DOWNSAMPLE = 64
VALID_MARGIN_PIXELS = 256
MIN_VALID_PATCH_RATIO = 0.10
NODATA_CANDIDATES = (0, 65535)

MIN_ROAD_COMPONENT_PIXELS = 80
MAX_HOLE_PIXELS = 500
CLOSING_ITERATIONS = 2

# All post-processing length parameters are metres, not fixed pixel counts.
MIN_CENTERLINE_COMPONENT_LENGTH_M = 4.0
MIN_CENTERLINE_SPUR_LENGTH_M = 6.0
MAX_CENTERLINE_GAP_M = 3.0
MAX_ENDPOINT_ANGLE_DEGREES = 25.0
SURFACE_GAP_SUPPORT_DILATION_PIXELS = 1

MIN_SURFACE_POLYGON_AREA_M2 = 8.0
MIN_CENTERLINE_LENGTH_M = 3.0
SIMPLIFY_TOLERANCE_M = 0.50
SAVE_FLOAT_PROBABILITY = False

print("Surface threshold:", SURFACE_THRESHOLD)
print("Center probability diagnostic threshold:", CENTERLINE_PROBABILITY_THRESHOLD)
print("Minimum center component length (m):", MIN_CENTERLINE_COMPONENT_LENGTH_M)


## Session 4 — ตรวจ GPU, ค้นหาภาพ และเลือก Checkpoint


In [ ]:
# ============================================================
# 4 — GPU, IMAGE DISCOVERY, CHECKPOINT SELECTION
# ============================================================

import re
import json
import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Prediction full-scene ต้องใช้ GPU กรุณาเลือก GPU Runtime")
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

EXCLUDE_IMAGE_KEYWORDS = {
    "mask", "label", "prob", "prediction", "pred", "clean",
    "centerline", "boundary", "road_surface", "output", "preview",
}


def collect_tiffs(sources):
    paths = []
    for src in sources:
        src = Path(src)
        if not src.exists():
            print("Path not found and skipped:", src)
            continue
        candidates = [src] if src.is_file() else src.rglob("*")
        for p in candidates:
            if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}:
                if not any(k in p.name.lower() for k in EXCLUDE_IMAGE_KEYWORDS):
                    paths.append(p)
    return sorted(set(paths), key=lambda p: p.name.lower())


def collect_checkpoints(sources):
    paths = []
    for src in sources:
        src = Path(src)
        if not src.exists():
            continue
        candidates = [src] if src.is_file() else src.rglob("*.pth")
        for p in candidates:
            if p.is_file() and p.suffix.lower() in {".pth", ".pt", ".ckpt"}:
                paths.append(p)
    return sorted(set(paths), key=lambda p: p.name.lower())


def checkpoint_rank(path):
    name_bonus = 1 if "best" in path.name.lower() else 0
    try:
        meta = torch.load(path, map_location="cpu", weights_only=False)
        score = float(meta.get("best_score", meta.get("val_score", -1.0)))
        epoch = int(meta.get("epoch", -1))
        arch_ok = 1 if meta.get("architecture_version") == "road_resnet34_unet_centerline_v2" else 0
        return (arch_ok, name_bonus, score, epoch, path.stat().st_mtime)
    except Exception as exc:
        print("Cannot inspect checkpoint:", path, repr(exc))
        return (0, name_bonus, -1.0, -1, path.stat().st_mtime)


IMAGE_PATHS = collect_tiffs(IMAGE_SOURCES)
CHECKPOINT_PATHS = collect_checkpoints(CHECKPOINT_SOURCES)

if not IMAGE_PATHS:
    raise FileNotFoundError("ไม่พบ THEOS-2 GeoTIFF ใน IMAGE_SOURCES")
if not CHECKPOINT_PATHS:
    raise FileNotFoundError("ไม่พบ checkpoint ใน CHECKPOINT_SOURCES")

CHECKPOINT_PATH = max(CHECKPOINT_PATHS, key=checkpoint_rank)
print("Images:")
for p in IMAGE_PATHS: print(" -", p)
print("Selected checkpoint:", CHECKPOINT_PATH)


## Session 5 — Model Architecture และโหลด Checkpoint

Model class ต้องตรงกับ Fine-tuning notebook ทุก layer ระบบจะตรวจ `architecture_version` ก่อนโหลดเพื่อป้องกัน silent mismatch


In [ ]:
# ============================================================
# 5 — MODEL ARCHITECTURE AND CHECKPOINT LOADING
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34


class ConvBNAct(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = ConvBNAct(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class RoadResNet34UNet(nn.Module):
    ARCHITECTURE_VERSION = "road_resnet34_unet_centerline_v2"

    def __init__(self, in_channels=4, out_channels=3):
        super().__init__()
        encoder = resnet34(weights=None)
        old_conv = encoder.conv1
        if in_channels != 3:
            encoder.conv1 = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False,
            )
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.enc1, self.enc2 = encoder.layer1, encoder.layer2
        self.enc3, self.enc4 = encoder.layer3, encoder.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 64)
        self.final_refine = ConvBNAct(64, 32)
        self.head = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        size = x.shape[-2:]
        s0 = self.stem(x)
        s1 = self.enc1(self.pool(s0))
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        b = self.enc4(s3)
        x = self.dec4(b, s3)
        x = self.dec3(x, s2)
        x = self.dec2(x, s1)
        x = self.dec1(x, s0)
        x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        return self.head(self.final_refine(x))


ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if ckpt.get("architecture_version") != RoadResNet34UNet.ARCHITECTURE_VERSION:
    raise RuntimeError(
        "Checkpoint ไม่ใช่ Centerline V2 (heatmap target): "
        f"{ckpt.get('architecture_version')}. Regenerate labels and retrain before prediction."
    )

config = ckpt.get("config", {})
BAND_INDEXES = tuple(config.get("band_indexes", [1, 2, 3, 4]))
HEAD_NAMES = config.get("head_names", ["surface", "centerline", "boundary"])
PATCH_SIZE = int(PATCH_SIZE_OVERRIDE or config.get("patch_size", 512))

model = RoadResNet34UNet(
    in_channels=int(config.get("in_channels", len(BAND_INDEXES))),
    out_channels=int(config.get("out_channels", 3)),
).to(DEVICE)

state = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
model.load_state_dict(state, strict=True)
model.eval()

print("Loaded checkpoint epoch:", ckpt.get("epoch"))
print("Best score:", ckpt.get("best_score"))
print("Patch size:", PATCH_SIZE)
print("Bands:", BAND_INDEXES)
print("Heads:", HEAD_NAMES)


## Session 6 — Image Audit, Valid BBox และ Normalization

ใช้ percentile 2–98 ของแต่ละภาพเพื่อลดปัญหาค่ารังสีต่างกันระหว่าง scene และป้องกัน NoData=65535 ดันค่าเฉลี่ยสูงผิดปกติ


In [ ]:
# ============================================================
# 6 — IMAGE AUDIT AND NORMALIZATION STATS
# ============================================================

import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window


def is_valid_rgb(rgb, nodata_value=None):
    valid = np.any(rgb > 0, axis=0)
    if nodata_value is not None:
        valid &= ~np.all(rgb == nodata_value, axis=0)
    for nd in NODATA_CANDIDATES:
        valid &= ~np.all(rgb == nd, axis=0)
    return valid


def find_valid_bbox_fast(image_path, downsample=64, margin=256):
    with rasterio.open(image_path) as src:
        if src.count < max(BAND_INDEXES):
            raise ValueError(f"{image_path.name} มี band ไม่ครบตาม checkpoint")
        out_h = max(int(np.ceil(src.height / downsample)), 1)
        out_w = max(int(np.ceil(src.width / downsample)), 1)
        rgb = src.read([1,2,3], out_shape=(3,out_h,out_w), resampling=Resampling.nearest)
        valid = is_valid_rgb(rgb, src.nodata)
        rows, cols = np.where(valid)
        if len(rows) == 0:
            raise RuntimeError(f"No valid RGB area: {image_path.name}")
        sx, sy = src.width/out_w, src.height/out_h
        left = max(int(np.floor(cols.min()*sx))-margin, 0)
        right = min(int(np.ceil((cols.max()+1)*sx))+margin, src.width)
        top = max(int(np.floor(rows.min()*sy))-margin, 0)
        bottom = min(int(np.ceil((rows.max()+1)*sy))+margin, src.height)
        return left, top, right, bottom


def compute_percentiles(image_path, bbox, max_size=1200):
    left, top, right, bottom = bbox
    width, height = right-left, bottom-top
    scale = max(width/max_size, height/max_size, 1)
    out_w, out_h = max(int(width/scale),1), max(int(height/scale),1)
    with rasterio.open(image_path) as src:
        arr = src.read(
            list(BAND_INDEXES), window=Window(left,top,width,height),
            out_shape=(len(BAND_INDEXES),out_h,out_w), resampling=Resampling.bilinear
        ).astype(np.float32)
        nodata = src.nodata
    lows, highs = [], []
    for band in arr:
        valid = np.isfinite(band)
        if nodata is not None: valid &= band != nodata
        for nd in NODATA_CANDIDATES: valid &= band != nd
        values = band[valid]
        if values.size < 100:
            lo, hi = 0.0, 1.0
        else:
            lo, hi = np.percentile(values,[2,98])
            if hi <= lo: hi = lo + 1.0
        lows.append(float(lo)); highs.append(float(hi))
    return np.asarray(lows,np.float32), np.asarray(highs,np.float32)


IMAGE_INFO = []
for image_path in IMAGE_PATHS:
    bbox = find_valid_bbox_fast(image_path, VALID_DOWNSAMPLE, VALID_MARGIN_PIXELS)
    lows, highs = compute_percentiles(image_path, bbox)
    with rasterio.open(image_path) as src:
        IMAGE_INFO.append({
            "path": image_path,
            "bbox": bbox,
            "lows": lows,
            "highs": highs,
            "crs": src.crs,
            "resolution": (abs(src.transform.a), abs(src.transform.e)),
        })
    print(image_path.name, "bbox=", bbox, "p02=", lows, "p98=", highs)


## Session 7 — Sliding-window Prediction แบบ Weighted Overlap

ใช้ memmap บน local disk เพื่อไม่ต้องถือ probability mosaic ทั้งหมดไว้ใน RAM และใช้ Hann-like weight ลดเส้นรอยต่อระหว่าง patch


In [ ]:
# ============================================================
# 7 — FULL-SCENE MULTI-HEAD PREDICTION
# ============================================================

import gc
import tempfile
from tqdm.auto import tqdm
from rasterio.windows import transform as window_transform


def positions_for_length(length, patch, stride):
    if length <= patch:
        return [0]
    pos = list(range(0, length-patch+1, stride))
    last = length-patch
    if pos[-1] != last: pos.append(last)
    return pos


def blend_weight(size):
    one = np.hanning(size).astype(np.float32)
    weight = np.outer(one, one)
    weight /= max(weight.max(), 1e-6)
    return np.clip(weight, 0.05, 1.0)


def normalize_patch(arr, lows, highs, nodata):
    arr = arr.astype(np.float32)
    valid = np.ones(arr.shape[1:], dtype=bool)
    if nodata is not None:
        valid &= ~np.all(arr == nodata, axis=0)
    for nd in NODATA_CANDIDATES:
        valid &= ~np.all(arr == nd, axis=0)
    out = (arr - lows[:,None,None]) / (highs[:,None,None] - lows[:,None,None] + 1e-6)
    out = np.clip(out, 0, 1)
    out[:, ~valid] = 0
    out[~np.isfinite(out)] = 0
    return out.astype(np.float32), valid


def amp_context():
    try:
        return torch.amp.autocast("cuda", enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.autocast(enabled=USE_AMP)


def predict_valid_bbox(info, output_dir):
    image_path = Path(info["path"])
    left, top, right, bottom = map(int, info["bbox"])
    width, height = right-left, bottom-top
    output_dir = Path(output_dir)
    geo_dir = output_dir / "geotiff"
    geo_dir.mkdir(parents=True, exist_ok=True)

    tmp_dir = Path(tempfile.mkdtemp(prefix=f"road_pred_{image_path.stem}_", dir="/content"))
    prob_path = tmp_dir / "prob_sum.dat"
    weight_path = tmp_dir / "weight_sum.dat"
    prob_sum = np.memmap(prob_path, mode="w+", dtype="float32", shape=(3,height,width))
    weight_sum = np.memmap(weight_path, mode="w+", dtype="float32", shape=(height,width))
    prob_sum[:] = 0; weight_sum[:] = 0

    xs = positions_for_length(width, PATCH_SIZE, STRIDE)
    ys = positions_for_length(height, PATCH_SIZE, STRIDE)
    tasks = [(x,y) for y in ys for x in xs]
    weight = blend_weight(PATCH_SIZE)

    with rasterio.open(image_path) as src:
        batch_arrays, batch_meta = [], []

        def flush_batch():
            if not batch_arrays: return
            tensor = torch.from_numpy(np.stack(batch_arrays)).to(DEVICE, non_blocking=True)
            with torch.no_grad(), amp_context():
                probs = torch.sigmoid(model(tensor)).float().cpu().numpy()
            for prob, (x,y,w,h) in zip(probs, batch_meta):
                ww = weight[:h,:w]
                prob_sum[:,y:y+h,x:x+w] += prob[:,:h,:w] * ww[None,:,:]
                weight_sum[y:y+h,x:x+w] += ww
            batch_arrays.clear(); batch_meta.clear()

        for x, y in tqdm(tasks, desc=f"Predict {image_path.name}"):
            w, h = min(PATCH_SIZE,width-x), min(PATCH_SIZE,height-y)
            src_win = Window(left+x, top+y, w, h)
            arr = src.read(list(BAND_INDEXES), window=src_win)
            norm, valid = normalize_patch(arr, info["lows"], info["highs"], src.nodata)
            if valid.mean() < MIN_VALID_PATCH_RATIO:
                continue
            padded = np.zeros((len(BAND_INDEXES),PATCH_SIZE,PATCH_SIZE), dtype=np.float32)
            padded[:,:h,:w] = norm
            batch_arrays.append(padded)
            batch_meta.append((x,y,w,h))
            if len(batch_arrays) >= BATCH_SIZE:
                flush_batch()
        flush_batch()

        crop_transform = window_transform(Window(left,top,width,height), src.transform)
        base_profile = src.profile.copy()
        prob_dtype = "float32" if SAVE_FLOAT_PROBABILITY else "uint8"
        prob_nodata = None if SAVE_FLOAT_PROBABILITY else 255
        base_profile.update(
            width=width, height=height, count=1, transform=crop_transform,
            dtype=prob_dtype, nodata=prob_nodata,
            compress="deflate", tiled=True, blockxsize=512, blockysize=512,
            BIGTIFF="IF_SAFER",
        )

    weight_safe = np.maximum(weight_sum, 1e-6)
    probability_paths = {}
    for head_index, head_name in enumerate(HEAD_NAMES):
        out_path = geo_dir / f"{image_path.stem}_{head_name}_probability.tif"
        with rasterio.open(out_path, "w", **base_profile) as dst:
            block = 1024
            for y in range(0,height,block):
                h = min(block,height-y)
                for x in range(0,width,block):
                    w = min(block,width-x)
                    p = prob_sum[head_index,y:y+h,x:x+w] / weight_safe[y:y+h,x:x+w]
                    if SAVE_FLOAT_PROBABILITY:
                        out = p.astype(np.float32)
                    else:
                        out = np.clip(np.round(p*100),0,100).astype(np.uint8)
                        out[weight_sum[y:y+h,x:x+w] <= 0] = 255
                    dst.write(out,1,window=Window(x,y,w,h))
        probability_paths[head_name] = out_path

    prob_sum.flush(); weight_sum.flush()
    del prob_sum, weight_sum
    gc.collect(); torch.cuda.empty_cache()
    return probability_paths, crop_transform, (width,height), tmp_dir


## Session 8 — Post-processing Surface, Centerline และ Boundary

ส่วนนี้หลีกเลี่ยง deprecated `skimage.remove_small_objects(min_size=...)` โดยใช้ `scipy.ndimage.label` โดยตรง

Centerline:

1. skeletonize surface mask
2. ใช้ centerline probability เป็น guide
3. fallback ไปใช้ skeleton ทั้งหมด หาก guide ตัดเส้นออกมากเกินไป

Boundary:

1. สร้าง geometric inner boundary จาก surface
2. ใช้ boundary probability เป็น guide
3. fallback หาก guide ไม่เพียงพอ


In [ ]:

# ============================================================
# 8 — POST-PROCESS RASTER OUTPUTS (CENTERLINE V2)
# ============================================================

from scipy.ndimage import label as ndi_label, binary_closing, binary_fill_holes, binary_dilation, binary_erosion
from scipy.spatial import cKDTree
from skimage.morphology import skeletonize
from skimage.draw import line as draw_line

EIGHT_CONNECTED = np.ones((3, 3), dtype=np.uint8)
NEIGHBORS = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]


def remove_small_components(mask, min_pixels):
    """8-connected removal; diagonal skeleton pixels are part of the same line."""
    labeled, count = ndi_label(mask, structure=EIGHT_CONNECTED)
    if count == 0:
        return mask.astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = sizes >= int(min_pixels)
    keep[0] = False
    return keep[labeled]


def fill_small_holes(mask, max_hole_pixels):
    filled = binary_fill_holes(mask)
    holes = filled & ~mask
    holes_labeled, count = ndi_label(holes, structure=EIGHT_CONNECTED)
    if count == 0:
        return mask
    sizes = np.bincount(holes_labeled.ravel())
    small = sizes <= int(max_hole_pixels)
    small[0] = False
    return mask | small[holes_labeled]


def read_probability(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        if src.dtypes[0] == "uint8" or arr.max(initial=0) > 1.5:
            arr = np.where(arr == 255, 0, arr / 100.0)
        return arr, src.profile.copy(), src.transform, src.crs


def save_uint8_mask(path, mask, profile):
    profile = profile.copy()
    profile.update(dtype="uint8", nodata=0, count=1, compress="deflate", predictor=1)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(mask.astype(np.uint8), 1)


def _pixel_neighbors(pixel, pixels):
    r, c = pixel
    return [(r + dr, c + dc) for dr, dc in NEIGHBORS if (r + dr, c + dc) in pixels]


def _edge_length_m(a, b, res_x_m, res_y_m):
    return float(np.hypot((a[0] - b[0]) * res_y_m, (a[1] - b[1]) * res_x_m))


def _skeleton_edge_length_m(mask, res_x_m, res_y_m):
    # Count each undirected edge once.  Correctly represents horizontal, vertical and diagonal paths.
    diagonal = float(np.hypot(res_x_m, res_y_m))
    return (
        float((mask[:, :-1] & mask[:, 1:]).sum()) * res_x_m
        + float((mask[:-1, :] & mask[1:, :]).sum()) * res_y_m
        + float((mask[:-1, :-1] & mask[1:, 1:]).sum()) * diagonal
        + float((mask[:-1, 1:] & mask[1:, :-1]).sum()) * diagonal
    )


def remove_short_components_by_length(mask, min_length_m, res_x_m, res_y_m):
    labels, count = ndi_label(mask, structure=EIGHT_CONNECTED)
    out = np.zeros_like(mask, dtype=bool)
    for component_id in range(1, count + 1):
        component = labels == component_id
        if _skeleton_edge_length_m(component, res_x_m, res_y_m) >= float(min_length_m):
            out |= component
    return out


def _trace_from_endpoint(endpoint, pixels):
    path = [endpoint]
    previous, current = None, endpoint
    while True:
        next_pixels = [q for q in _pixel_neighbors(current, pixels) if q != previous]
        if not next_pixels:
            break
        nxt = next_pixels[0]
        path.append(nxt)
        previous, current = current, nxt
        if len(_pixel_neighbors(current, pixels)) != 2:
            break
    return path


def prune_short_spurs(mask, min_spur_length_m, res_x_m, res_y_m, max_rounds=20):
    """Remove only endpoint-to-junction twigs; never delete a full through-road."""
    result = mask.astype(bool).copy()
    for _ in range(max_rounds):
        pixels = set(map(tuple, np.argwhere(result)))
        if not pixels:
            break
        degrees = {p: len(_pixel_neighbors(p, pixels)) for p in pixels}
        remove = set()
        for endpoint in (p for p, degree in degrees.items() if degree == 1):
            path = _trace_from_endpoint(endpoint, pixels)
            end_degree = degrees.get(path[-1], 0)
            length_m = sum(_edge_length_m(a, b, res_x_m, res_y_m) for a, b in zip(path, path[1:]))
            if end_degree >= 3 and length_m < float(min_spur_length_m):
                remove.update(path[:-1])  # retain the junction itself
        if not remove:
            break
        rr, cc = zip(*remove)
        result[np.asarray(rr), np.asarray(cc)] = False
    return result


def _endpoint_inward_vector(endpoint, pixels, res_x_m, res_y_m, steps=5):
    path = _trace_from_endpoint(endpoint, pixels)
    point = path[min(len(path) - 1, steps)]
    vec = np.array([(point[0] - endpoint[0]) * res_y_m, (point[1] - endpoint[1]) * res_x_m], dtype=np.float32)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 1e-6 else None


def bridge_collinear_endpoint_gaps(mask, surface, center_prob, res_x_m, res_y_m):
    """Conservatively bridge short, aligned endpoint gaps fully supported by surface.

    Centerline probability only ranks otherwise valid candidates. It is never a hard gate,
    so a weak learned head cannot fragment the surface-derived skeleton.
    """
    if MAX_CENTERLINE_GAP_M <= 0:
        return mask.astype(bool)
    result = mask.astype(bool).copy()
    support = binary_dilation(surface.astype(bool), iterations=int(SURFACE_GAP_SUPPORT_DILATION_PIXELS))
    cos_limit = float(np.cos(np.deg2rad(MAX_ENDPOINT_ANGLE_DEGREES)))

    for _ in range(20):
        pixels = set(map(tuple, np.argwhere(result)))
        if len(pixels) < 2:
            break
        degrees = {p: len(_pixel_neighbors(p, pixels)) for p in pixels}
        endpoints = [p for p, degree in degrees.items() if degree == 1]
        if len(endpoints) < 2:
            break
        labels, _ = ndi_label(result, structure=EIGHT_CONNECTED)
        coords_m = np.array([[p[0] * res_y_m, p[1] * res_x_m] for p in endpoints], dtype=np.float32)
        candidate_pairs = cKDTree(coords_m).query_pairs(r=float(MAX_CENTERLINE_GAP_M))
        candidates = []
        for ia, ib in candidate_pairs:
            a, b = endpoints[ia], endpoints[ib]
            if labels[a] == labels[b]:
                continue
            inward_a = _endpoint_inward_vector(a, pixels, res_x_m, res_y_m)
            inward_b = _endpoint_inward_vector(b, pixels, res_x_m, res_y_m)
            direction = coords_m[ib] - coords_m[ia]
            direction /= max(np.linalg.norm(direction), 1e-6)
            if inward_a is None or inward_b is None:
                continue
            if float(np.dot(-inward_a, direction)) < cos_limit:
                continue
            if float(np.dot(inward_b, direction)) < cos_limit:
                continue
            rr, cc = draw_line(a[0], a[1], b[0], b[1])
            if not support[rr, cc].all():
                continue
            # Higher learned probability is preferred only when geometric conditions already pass.
            candidates.append((float(center_prob[rr, cc].mean()), rr, cc))
        if not candidates:
            break
        _, rr, cc = max(candidates, key=lambda item: item[0])
        result[rr, cc] = True
    return result


def postprocess_probabilities(probability_paths, output_dir):
    surface_prob, profile, transform, crs = read_probability(probability_paths["surface"])
    center_prob, _, _, _ = read_probability(probability_paths["centerline"])
    boundary_prob, _, _, _ = read_probability(probability_paths["boundary"])
    res_x_m, res_y_m = abs(float(transform.a)), abs(float(transform.e))
    if crs is None or crs.is_geographic:
        raise RuntimeError("Prediction output must use a projected CRS in metres for centreline length cleanup.")

    surface = surface_prob >= SURFACE_THRESHOLD
    surface = remove_small_components(surface, MIN_ROAD_COMPONENT_PIXELS)
    if CLOSING_ITERATIONS > 0:
        surface = binary_closing(surface, iterations=CLOSING_ITERATIONS)
    surface = fill_small_holes(surface, MAX_HOLE_PIXELS)
    surface = remove_small_components(surface, MIN_ROAD_COMPONENT_PIXELS)

    # The surface skeleton is authoritative.  Center probability is a confidence/ranking signal only.
    center = skeletonize(surface)
    center = prune_short_spurs(center, MIN_CENTERLINE_SPUR_LENGTH_M, res_x_m, res_y_m)
    center = remove_short_components_by_length(center, MIN_CENTERLINE_COMPONENT_LENGTH_M, res_x_m, res_y_m)
    center = bridge_collinear_endpoint_gaps(center, surface, center_prob, res_x_m, res_y_m)
    center = prune_short_spurs(center, MIN_CENTERLINE_SPUR_LENGTH_M, res_x_m, res_y_m)
    center = remove_short_components_by_length(center, MIN_CENTERLINE_COMPONENT_LENGTH_M, res_x_m, res_y_m)

    geometric_boundary = surface & ~binary_erosion(surface, iterations=1, border_value=0)
    boundary_guide = binary_dilation(boundary_prob >= BOUNDARY_THRESHOLD, iterations=3)
    boundary = geometric_boundary & boundary_guide
    if geometric_boundary.sum() > 0 and boundary.sum() / geometric_boundary.sum() < 0.35:
        print("Boundary guide retained too little; fallback to geometric boundary")
        boundary = geometric_boundary

    geo_dir = Path(output_dir) / "geotiff"
    paths = {
        "surface": geo_dir / "road_surface_mask.tif",
        "centerline": geo_dir / "road_centerline_mask.tif",
        "boundary": geo_dir / "road_boundary_mask.tif",
    }
    save_uint8_mask(paths["surface"], surface, profile)
    save_uint8_mask(paths["centerline"], center, profile)
    save_uint8_mask(paths["boundary"], boundary, profile)
    return surface, center, boundary, center_prob, paths, profile, transform, crs


## Session 9 — Vectorization เป็น GeoPackage

- Surface ใช้ raster polygonization
- Boundary ใช้ polygon boundary จึงได้เส้นข้างถนน
- Centerline trace จาก skeleton pixels เป็น LineString segments
- ไม่สร้าง node/edge layers
- คำนวณความกว้างประมาณจาก `2 × distance-to-boundary` ที่ centerline pixels


In [ ]:
# ============================================================
# 9 — VECTOR EXPORT: SURFACE, CENTERLINE, BOUNDARY
# ============================================================

import fiona
import geopandas as gpd
from shapely.geometry import shape, LineString, Polygon, MultiPolygon, MultiLineString
from rasterio.features import shapes
from scipy.ndimage import distance_transform_edt

NEIGHBORS = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]


def write_empty_gpkg(path, layer, geometry_type, crs, properties):
    path = Path(path)
    if path.exists(): path.unlink()
    schema = {"geometry": geometry_type, "properties": properties}
    kwargs = {"driver":"GPKG", "schema":schema, "layer":layer}
    if crs is not None: kwargs["crs_wkt"] = crs.to_wkt()
    with fiona.open(path, "w", **kwargs):
        pass


def polygonize_surface(surface, transform, crs, min_area_m2):
    geoms, areas = [], []
    for geom, value in shapes(surface.astype(np.uint8), mask=surface, transform=transform):
        if int(value) != 1: continue
        poly = shape(geom)
        if not poly.is_valid: poly = poly.buffer(0)
        if poly.is_empty: continue
        pieces = list(poly.geoms) if isinstance(poly, MultiPolygon) else [poly]
        for p in pieces:
            if p.area >= min_area_m2:
                p = p.simplify(SIMPLIFY_TOLERANCE_M, preserve_topology=True)
                geoms.append(p); areas.append(float(p.area))
    return gpd.GeoDataFrame(
        {"surface_id": range(1,len(geoms)+1), "area_m2": areas},
        geometry=geoms, crs=crs,
    )


def skeleton_paths(skeleton):
    pixels = set(map(tuple, np.argwhere(skeleton)))
    if not pixels: return []

    def neighbors(p):
        r,c = p
        return [(r+dr,c+dc) for dr,dc in NEIGHBORS if (r+dr,c+dc) in pixels]

    degree = {p: len(neighbors(p)) for p in pixels}
    nodes = {p for p,d in degree.items() if d != 2}
    visited = set()
    paths = []

    def edge_key(a,b): return tuple(sorted((a,b)))

    for start in list(nodes):
        for nxt in neighbors(start):
            key = edge_key(start,nxt)
            if key in visited: continue
            path = [start,nxt]
            visited.add(key)
            prev, cur = start, nxt
            while cur not in nodes:
                options = [q for q in neighbors(cur) if q != prev and edge_key(cur,q) not in visited]
                if not options: break
                q = options[0]
                visited.add(edge_key(cur,q))
                path.append(q)
                prev, cur = cur, q
            if len(path) >= 2: paths.append(path)

    # handle closed loops without endpoints/junctions
    for start in list(pixels):
        unused = [q for q in neighbors(start) if edge_key(start,q) not in visited]
        if not unused: continue
        path = [start, unused[0]]
        visited.add(edge_key(start, unused[0]))
        prev, cur = start, unused[0]
        while True:
            options = [q for q in neighbors(cur) if q != prev and edge_key(cur,q) not in visited]
            if not options: break
            q = options[0]
            visited.add(edge_key(cur,q))
            path.append(q)
            prev, cur = cur, q
            if cur == start: break
        if len(path) >= 3: paths.append(path)
    return paths


def pixel_path_to_line(path, transform):
    coords = []
    for row,col in path:
        x,y = transform * (col+0.5, row+0.5)
        coords.append((x,y))
    if len(set(coords)) < 2: return None
    return LineString(coords).simplify(SIMPLIFY_TOLERANCE_M, preserve_topology=True)


def centerline_gdf(center, surface, center_prob, transform, crs):
    res_x, res_y = abs(transform.a), abs(transform.e)
    distance_m = distance_transform_edt(surface, sampling=(res_y,res_x))
    geoms, lengths, med_widths, mean_widths, confidences = [],[],[],[],[]
    for path in skeleton_paths(center):
        line = pixel_path_to_line(path, transform)
        if line is None or line.is_empty or line.length < MIN_CENTERLINE_LENGTH_M:
            continue
        rows = np.asarray([p[0] for p in path], dtype=int)
        cols = np.asarray([p[1] for p in path], dtype=int)
        widths = 2.0 * distance_m[rows,cols]
        geoms.append(line)
        lengths.append(float(line.length))
        med_widths.append(float(np.median(widths)) if len(widths) else np.nan)
        mean_widths.append(float(np.mean(widths)) if len(widths) else np.nan)
        confidences.append(float(np.mean(center_prob[rows, cols])) if len(rows) else 0.0)
    return gpd.GeoDataFrame({
        "center_id": range(1,len(geoms)+1),
        "length_m": lengths,
        "width_med_m": med_widths,
        "width_mean_m": mean_widths,
        "confidence": confidences,
    }, geometry=geoms, crs=crs)


def boundary_gdf(surface_gdf):
    geoms = []
    for geom in surface_gdf.geometry:
        boundary = geom.boundary
        parts = list(boundary.geoms) if isinstance(boundary, MultiLineString) else [boundary]
        geoms.extend([g for g in parts if g is not None and not g.is_empty])
    return gpd.GeoDataFrame({
        "boundary_id": range(1,len(geoms)+1),
        "length_m": [float(g.length) for g in geoms],
    }, geometry=geoms, crs=surface_gdf.crs)


def export_vectors(surface, center, center_prob, transform, crs, output_dir):
    vector_dir = Path(output_dir) / "vector"
    vector_dir.mkdir(parents=True, exist_ok=True)
    surface_path = vector_dir / "road_surface.gpkg"
    center_path = vector_dir / "road_centerline.gpkg"
    boundary_path = vector_dir / "road_boundary.gpkg"

    surface_gdf = polygonize_surface(surface, transform, crs, MIN_SURFACE_POLYGON_AREA_M2)
    if len(surface_gdf):
        if surface_path.exists(): surface_path.unlink()
        surface_gdf.to_file(surface_path, driver="GPKG", layer="road_surface", index=False)
    else:
        write_empty_gpkg(surface_path, "road_surface", "Polygon", crs, {"surface_id":"int","area_m2":"float"})

    center_gdf = centerline_gdf(center, surface, center_prob, transform, crs)
    if len(center_gdf):
        if center_path.exists(): center_path.unlink()
        center_gdf.to_file(center_path, driver="GPKG", layer="road_centerline", index=False)
    else:
        write_empty_gpkg(center_path, "road_centerline", "LineString", crs, {
            "center_id":"int","length_m":"float","width_med_m":"float","width_mean_m":"float","confidence":"float"
        })

    boundary_frame = boundary_gdf(surface_gdf) if len(surface_gdf) else gpd.GeoDataFrame(geometry=[],crs=crs)
    if len(boundary_frame):
        if boundary_path.exists(): boundary_path.unlink()
        boundary_frame.to_file(boundary_path, driver="GPKG", layer="road_boundary", index=False)
    else:
        write_empty_gpkg(boundary_path, "road_boundary", "LineString", crs, {"boundary_id":"int","length_m":"float"})

    return {
        "surface_gpkg": surface_path,
        "centerline_gpkg": center_path,
        "boundary_gpkg": boundary_path,
        "surface_features": len(surface_gdf),
        "centerline_features": len(center_gdf),
        "boundary_features": len(boundary_frame),
    }


## Session 10 — รัน Pipeline กับภาพทั้งหมด

แต่ละภาพถูกประมวลผลแยกกันและล้าง temporary memmap หลังจบ เพื่อควบคุม RAM และ disk usage


In [ ]:
# ============================================================
# 10 — RUN ALL IMAGES
# ============================================================

import shutil
import gc

RESULTS = []
for info in IMAGE_INFO:
    image_path = Path(info["path"])
    image_out = OUTPUT_ROOT / image_path.stem
    image_out.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print("Processing:", image_path.name)
    print("#" * 100)

    probability_paths, crop_transform, crop_size, tmp_dir = predict_valid_bbox(info, image_out)
    surface, center, boundary, center_prob, mask_paths, profile, transform, crs = postprocess_probabilities(
        probability_paths, image_out
    )
    vector_result = export_vectors(surface, center, center_prob, transform, crs, image_out)

    result = {
        "image": str(image_path),
        "checkpoint": str(CHECKPOINT_PATH),
        "bbox": list(map(int, info["bbox"])),
        "surface_probability": str(probability_paths["surface"]),
        "centerline_probability": str(probability_paths["centerline"]),
        "boundary_probability": str(probability_paths["boundary"]),
        "surface_mask": str(mask_paths["surface"]),
        "centerline_mask": str(mask_paths["centerline"]),
        "boundary_mask": str(mask_paths["boundary"]),
        **{k: str(v) if isinstance(v, Path) else v for k,v in vector_result.items()},
    }
    RESULTS.append(result)

    shutil.rmtree(tmp_dir, ignore_errors=True)
    del surface, center, boundary
    gc.collect(); torch.cuda.empty_cache()

summary_df = pd.DataFrame(RESULTS)
summary_csv = OUTPUT_ROOT / "road_prediction_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved summary:", summary_csv)
display(summary_df)


## Session 11 — Preview ผลลัพธ์

สีใน overlay:

- ม่วงโปร่งใส = road surface
- เขียว = centerline
- เหลือง = boundary


In [ ]:
# ============================================================
# 11 — RESULT PREVIEW
# ============================================================

import matplotlib.pyplot as plt
from rasterio.enums import Resampling


def read_rgb_preview(image_path, bbox, max_size=1000):
    left,top,right,bottom = bbox
    width,height = right-left,bottom-top
    scale = max(width/max_size,height/max_size,1)
    out_w,out_h = max(int(width/scale),1),max(int(height/scale),1)
    with rasterio.open(image_path) as src:
        rgb = src.read([1,2,3],window=Window(left,top,width,height),out_shape=(3,out_h,out_w),resampling=Resampling.bilinear).astype(np.float32)
        nodata = src.nodata
    out = np.zeros_like(rgb,dtype=np.float32)
    for i in range(3):
        valid = np.isfinite(rgb[i])
        if nodata is not None: valid &= rgb[i] != nodata
        for nd in NODATA_CANDIDATES: valid &= rgb[i] != nd
        vals = rgb[i][valid]
        if len(vals):
            lo,hi = np.percentile(vals,[2,98]); hi=max(hi,lo+1)
            out[i]=np.clip((rgb[i]-lo)/(hi-lo),0,1)
    return np.transpose(out,(1,2,0))


def read_mask_preview(path, max_size=1000):
    with rasterio.open(path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1)

        out_h = max(int(src.height / scale), 1)
        out_w = max(int(src.width / scale), 1)

        # average ใช้กับ rasterio.read ได้
        # out_dtype=float32 ทำให้เส้น binary บาง ๆ ยังมีค่ามากกว่า 0 หลังย่อภาพ
        return src.read(
            1,
            out_shape=(out_h, out_w),
            out_dtype="float32",
            resampling=Resampling.average,
        )

for row in RESULTS:
    bbox = tuple(row["bbox"])
    rgb = read_rgb_preview(Path(row["image"]),bbox)
    surface = read_mask_preview(row["surface_mask"])
    center = read_mask_preview(row["centerline_mask"])
    boundary = read_mask_preview(row["boundary_mask"])

    overlay = rgb.copy()
    overlay[surface>0] = 0.65*overlay[surface>0] + 0.35*np.array([1.0,0.0,1.0])
    overlay[boundary>0] = np.array([1.0,0.8,0.0])
    overlay[center>0] = np.array([0.0,1.0,0.0])

    fig,axes=plt.subplots(1,4,figsize=(22,6))
    axes[0].imshow(rgb); axes[0].set_title("THEOS-2 RGB")
    axes[1].imshow(surface,cmap="gray"); axes[1].set_title("Road surface")
    axes[2].imshow(center,cmap="gray"); axes[2].set_title("Centerline")
    axes[3].imshow(overlay); axes[3].set_title("Surface + boundary + centerline")
    for ax in axes: ax.axis("off")
    plt.suptitle(Path(row["image"]).name)
    plt.tight_layout(); plt.show()


## Session 12 — สรุป Output

เปิดไฟล์ `.gpkg` ใน QGIS ได้โดยตรง

- `road_surface.gpkg` = polygon ถนน
- `road_centerline.gpkg` = เส้นกึ่งกลางพร้อมความกว้างประมาณ
- `road_boundary.gpkg` = เส้นข้างถนน

หากผลถนนหนา/บางเกินไป ให้ปรับ threshold ใน Session 3 แล้วรันตั้งแต่ Session 7 ใหม่ โดยไม่ต้องโหลด checkpoint ใหม่


In [ ]:
# ============================================================
# 12 — FINAL SUMMARY
# ============================================================

print("All outputs:", OUTPUT_ROOT)
print("Summary CSV:", summary_csv)
for row in RESULTS:
    print("\n", "="*80)
    print("Image:", Path(row["image"]).name)
    print("Surface GPKG   :", row["surface_gpkg"])
    print("Centerline GPKG:", row["centerline_gpkg"])
    print("Boundary GPKG  :", row["boundary_gpkg"])
    print("Surface features:", row["surface_features"])
    print("Center features :", row["centerline_features"])
    print("Boundary features:", row["boundary_features"])


## เอกสารอ้างอิงทางเทคนิค

- Rasterio windowed reading/writing: https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html
- Rasterio vector features: https://rasterio.readthedocs.io/en/stable/topics/features.html
- Torchvision ResNet34: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html
